# BE-03 · Requests, responses, serialization, validation, status codes

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/BE-03.ipynb)

**Code:** [kaizen/server-ktor](https://github.com/BuiltBySwap/kaizen/tree/main/server-ktor)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: BE-03
title: "Requests, responses, serialization, validation, status codes"
track: backend
date: 2026-10-15
code: "kaizen/server-ktor"
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "An API is a contract with phones you cannot update: only add fields, be a tolerant reader, answer every client mistake with an explicit 4xx (400 unreadable, 422 breaks a rule, 409 conflicts, 415 wrong type) and keep 500 for real bugs."
tags: [BE-03, backend, ktor, api, validation, status-codes, serialization, backward-compatibility]
pending:
  - "English: 3 sentences with latency and throughput (past simple vs present perfect)"
  - "CTO lens: the business impact of API contracts"
  - "MQ423 in my own words"
  - "Note the original 'but was' status of the failing 415 test (my guess: 500)"
```

## Goal
Make the Kaizen API safe to call from phones I cannot update: a clear contract, JSON that does not break old apps, validation at the edge, and the right status code for every failure.

## Why: an API is a contract with phones you cannot update
On a web page a fix reaches everyone on refresh. A phone somewhere runs the app from **two years ago** and still calls my server. If I rename `mobileQuestion` to `question`, that app finds nothing and breaks, and I cannot fix it from the server.

## MQ423 · How does an API stay backward compatible with 2-year-old app versions?
| Rule | Why |
|---|---|
| **Only add, never remove or rename** a field | Old apps still find what they read |
| **New request fields get defaults or are optional** | Old apps do not send them and must still work |
| **Never change a field's type** (`"5"` to `5`) | The old parser fails |
| **Be a tolerant reader:** clients ignore fields they do not know | New response fields must not crash old apps |
| **Version only as a last resort** (`/v2`) | I would then run two versions for years |
| **Deprecate with a window and keep a minimum-supported-version gate** | Eventually very old apps are told "please update" |
| **Protect it with contract tests** | A change that breaks the contract turns a test red |

**The Kotlin catch:** kotlinx.serialization's `Json` **fails on unknown keys by default**, so the Android client needs `ignoreUnknownKeys = true`. Also, properties equal to their default are **not written** unless `encodeDefaults = true`, so the response shape would be unstable.

In [ ]:
def breaking_changes(old, new, kind="response"):
    """old / new: field name -> {"type": str, "required": bool}. Returns what would break OLD apps."""
    problems = []
    if kind == "response":                       # old apps READ these fields
        for name, spec in old.items():
            if name not in new:
                problems.append(f"removed field '{name}'")
            elif new[name]["type"] != spec["type"]:
                problems.append(f"'{name}' changed type {spec['type']} -> {new[name]['type']}")
    else:                                        # old apps SEND these fields
        for name, spec in new.items():
            if name not in old and spec["required"]:
                problems.append(f"new REQUIRED field '{name}': old apps never send it")
    return problems

v1 = {"date": {"type": "string", "required": True}, "mobileQuestion": {"type": "string", "required": True}}

renamed = {"date": v1["date"], "question": {"type": "string", "required": True}}                # rename = remove + add
added   = {**v1, "tags": {"type": "list", "required": False}}                                   # adding is safe
retyped = {"date": v1["date"], "mobileQuestion": {"type": "int", "required": True}}

print("rename a field      :", breaking_changes(v1, renamed))
print("add a field         :", breaking_changes(v1, added))
print("change a type       :", breaking_changes(v1, retyped))
print("request: add REQUIRED:", breaking_changes(v1, {**v1, "mood": {"type": "string", "required": True}}, "request"))
print("request: add optional:", breaking_changes(v1, {**v1, "mood": {"type": "string", "required": False}}, "request"))
assert breaking_changes(v1, renamed) == ["removed field 'mobileQuestion'"]
assert breaking_changes(v1, added) == []
assert len(breaking_changes(v1, retyped)) == 1
assert len(breaking_changes(v1, {**v1, "mood": {"type": "string", "required": True}}, "request")) == 1
assert breaking_changes(v1, {**v1, "mood": {"type": "string", "required": False}}, "request") == []

## What happens on `POST /progress`
```text
phone ──POST /progress  {"date":"2026-10-15","status":"done"}──▶ Ktor
   0. my own check        Content-Type must be application/json          (otherwise 415)
   1. ContentNegotiation  JSON text → ProgressRequest                    (not valid JSON, or wrong shape → 400)
   2. RequestValidation   do the values follow the rules?                (status not allowed, bad date → 422)
   3. my handler          create it, or find a conflict                  (already exists → 409)
   4. respond             201 Created + the new object + a Location header
```
`StatusPages` is the one place that turns an exception from step 1 or 2 into a response with the standard `{"error": ...}` shape.

## Status codes
| Code | Meaning | Kaizen example |
|---|---|---|
| **200 OK** | Success with a body | `GET /plan/2026-10-11` |
| **201 Created** | A new thing exists; add a `Location` header | `POST /progress` |
| **204 No Content** | Success, nothing to send back | a delete |
| **400 Bad Request** | The request cannot be **understood** | body is `{ not json` |
| **401 Unauthorized** | Not logged in (BE-06) | |
| **403 Forbidden** | Logged in but not allowed | |
| **404 Not Found** | Valid request, nothing there | `/plan/2030-01-01` |
| **409 Conflict** | Clashes with the current state | progress for that date already exists |
| **415 Unsupported Media Type** | The body is not in a format the endpoint accepts | a `text/plain` body |
| **422 Unprocessable Content** | Understood and well-formed, but **breaks a rule** | `"status": "banana"` |
| **500 Server Error** | A bug on **our** side | an unhandled exception |

**400 vs 422:** 400 means "I cannot read this"; 422 means "I can read it, but it is not allowed". A 422 can show "fix field X"; a 400 means the app has a bug. Keep **5xx for real bugs only**: a 5xx tells monitoring "our server is broken".

In [ ]:
import datetime, json, re

ALLOWED = {"not-started", "in-progress", "partial", "done", "skipped"}

def post_progress(content_type, body, existing):
    """The decision order of POST /progress, as plain Python."""
    if content_type != "application/json":
        return 415
    try:
        data = json.loads(body)
    except ValueError:
        return 400                                            # not valid JSON
    if not isinstance(data, dict) or "date" not in data or "status" not in data:
        return 400                                            # cannot become a ProgressRequest
    problems = []
    if data["status"] not in ALLOWED:
        problems.append("status")
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(data["date"])):
        problems.append("date")
    else:
        try:
            datetime.date.fromisoformat(data["date"])
        except ValueError:
            problems.append("date")
    if len(data.get("note", "")) > 500:
        problems.append("note")
    if problems:
        return 422                                            # readable, but breaks a rule
    if data["date"] in existing:
        return 409                                            # clashes with current state
    existing.add(data["date"])
    return 201

seen = set()
cases = [
    ("application/json", '{"date":"2026-10-15","status":"done"}', 201),
    ("application/json", '{"date":"2026-10-15","status":"done"}', 409),   # the same date again
    ("application/json", '{"date":"2026-10-15","status":"banana"}', 422),
    ("application/json", '{"date":"tomorrow","status":"done"}', 422),
    ("application/json", '{"date":"2026-02-30","status":"done"}', 422),   # a well-formed but impossible date
    ("application/json", '{ not json', 400),
    ("application/json", '{"date":"2026-10-16"}', 400),                   # a required field is missing
    ("text/plain", "hello", 415),
    ("application/json", '{"date":"2026-10-17","status":"done","addedLater":"x"}', 201),   # tolerant reader
]
for content_type, body, expected in cases:
    got = post_progress(content_type, body, seen)
    print(f"{got}  expected {expected}  {body[:48]}")
    assert got == expected

## The Ktor code, file by file

**Four ideas.** *Application* is the server object (your setup functions are extension functions on it). A *plugin* (`install`) wraps every call, like an OkHttp interceptor for incoming requests. *Routing* maps (method, path) to a handler. `call` is one request plus its response.

**When does a line run?** Once at **startup**: the module functions, `install(...)`, the shape of `routing { }`, loading the CSV. On **every request**: only the code inside a handler `{ ... }`, which is a suspend lambda and so runs in a coroutine.

**Dependencies in `server-ktor/build.gradle.kts`:**
| Line | What it gives |
|---|---|
| `ktorLibs.server.core` | `Application`, `call`, `routing`, `respond`, parameters |
| `ktorLibs.server.netty` | The engine: sockets and threads |
| `ktorLibs.server.config.yaml` | Lets Ktor read `application.yaml` (in Ktor 3 it is its own module) |
| `ktorLibs.server.contentNegotiation` | Converts objects ↔ JSON |
| `ktorLibs.serialization.kotlinx.json` | The JSON format it uses |
| `ktorLibs.server.callLogging` | Request logging |
| `ktorLibs.server.requestValidation` | Rule checks on request bodies |
| `ktorLibs.server.statusPages` | Exception → response |
| `libs.logback.classic` | The logging backend |

`ktorLibs` is Ktor's own version catalog (all modules share one version); `libs` is the project's `libs.versions.toml`. A dependency only makes a plugin **available**; it works only when `install(...)` runs and the module is listed in `application.yaml`.

**`Models.kt` (added):**
```kotlin
@Serializable data class FieldError(val field: String, val message: String)
@Serializable data class ProgressRequest(val date: String, val status: String, val note: String = "")
@Serializable data class Progress(val date: String, val status: String, val note: String, val createdAt: String)
@Serializable data class ApiError(val error: String, val code: String = "error", val fields: List<FieldError> = emptyList())
```
`note` has a default, so old apps that never send it still work. `ApiError` gained `code` and `fields` **without breaking** any existing `ApiError("...")` call: add, do not rename. `code` is for apps to switch on; the message can change, the code is a promise. `status` is a `String` on purpose: a Kotlin enum would reject a bad value while parsing (a 400) and validation would never run; the enum is often the better real-world choice, a trade-off to say out loud.

**`Serialization.kt`:** `json(Json { ignoreUnknownKeys = true; encodeDefaults = true })`.

**`Validation.kt`:** `install(RequestValidation)` with `validate<ProgressRequest> { ... }` collects **all** problems (status, date, note) and returns `ValidationResult.Invalid(problems)`. `install(StatusPages)` maps `RequestValidationException` to **422**, `UnsupportedMediaTypeException` to **415**, and the catch-all `Throwable` to **400** (unreadable body) or **500** (a real bug, logged on our side; the client only sees "internal error", because a stack trace would leak how the server works). Validation runs **when the handler calls `call.receive<ProgressRequest>()`**: Ktor converts the JSON, runs the rules, and throws if one fails. The module is registered in `application.yaml` **before** routing.

**`ProgressStore.kt`:**
```kotlin
class ProgressStore {
    private val byDate = ConcurrentHashMap<String, Progress>()   // requests run on several threads at once
    fun create(request: ProgressRequest): Progress? {
        val progress = Progress(request.date, request.status, request.note, Instant.now().toString())
        return if (byDate.putIfAbsent(request.date, progress) == null) progress else null   // null = conflict
    }
    fun find(date: String): Progress? = byDate[date]
}
```
`putIfAbsent` checks and inserts as one indivisible step, so two simultaneous posts for the same date cannot both succeed.

**`Routing.kt` (added):**
```kotlin
post("/progress") {
    if (call.rejectIfNotJson()) return@post                   // 415, decided explicitly by us
    val request = call.receive<ProgressRequest>()             // JSON -> object; the rules run here
    val created = progressStore.create(request)
    if (created == null) { call.respond(HttpStatusCode.Conflict, ApiError("progress for ${request.date} already exists", "conflict")); return@post }
    call.response.header(HttpHeaders.Location, "/progress/${created.date}")
    call.respond(HttpStatusCode.Created, created)             // 201
}
get("/progress/{date}") { /* 200 or 404 */ }

private suspend fun ApplicationCall.rejectIfNotJson(): Boolean {
    if (request.contentType().withoutParameters() == ContentType.Application.Json) return false
    respond(HttpStatusCode.UnsupportedMediaType, ApiError("send the body as application/json", "unsupported_media_type"))
    return true
}
```
**Tests (`ServerTest.kt`, 17 in total):** each starts the whole app in memory with `testApplication { configure() }`: no port, no stale server. They cover 200/400/404 for plans and questions, and for `/progress`: 201 with a `Location` header, 409, 422 (unknown status, bad date), 400 (broken JSON, missing required field), 415, the tolerant reader, the default for a missing `note`, and reading back.

**`requests.http`:** one request per case (health, plan, create, duplicate, unknown status, broken JSON, not JSON, read back) for IntelliJ's HTTP client.

In [ ]:
%%kotlin
import java.time.LocalDate
import kotlin.test.assertEquals

val ALLOWED_STATUSES = setOf("not-started", "in-progress", "partial", "done", "skipped")

// the same rules as Validation.kt, without Ktor, so they can be tested on their own
fun problemsWith(date: String, status: String, note: String = ""): List<String> {
    val problems = mutableListOf<String>()
    if (status !in ALLOWED_STATUSES) problems += "status: must be one of ${ALLOWED_STATUSES.joinToString()}"
    if (runCatching { LocalDate.parse(date) }.isFailure) problems += "date: must look like 2026-10-15"
    if (note.length > 500) problems += "note: must be at most 500 characters"
    return problems
}

assertEquals(emptyList(), problemsWith("2026-10-15", "done"))
assertEquals(listOf("status"), problemsWith("2026-10-15", "banana").map { it.substringBefore(":") })
assertEquals(listOf("date", "status"), problemsWith("tomorrow", "banana").map { it.substringBefore(":") }.sorted())   // ALL problems at once
assertEquals(listOf("date"), problemsWith("2026-02-30", "done").map { it.substringBefore(":") })                       // well-formed, impossible
assertEquals(listOf("note"), problemsWith("2026-10-15", "done", "x".repeat(501)).map { it.substringBefore(":") })
assertEquals(emptyList(), problemsWith("2026-10-15", "done", "x".repeat(500)))
println("validation rules: 6 checks passed")

## Bug diary
| Error or symptom | Cause | Fix |
|---|---|---|
| `ServerTest > test root endpoint FAILED` after Step 3 | The generator's sample test asked for `GET /`, which I removed in PRJ-02. A **stale test**, not a code bug | Replace it with tests of the real contract. Update the test to match behaviour chosen on purpose; do not delete or revert blindly |
| `a body that is not JSON returns 415` FAILED (the other 16 passed) | `ContentNegotiation` cannot convert a `text/plain` body. Ktor throws an exception my catch-all did not recognise, so my guess is that it answered **500**: a client mistake looking like a server bug | Check the `Content-Type` **explicitly** before `receive`, so the contract does not depend on accidental exception types |
| A dependency list that did not match mine | I wrote it from memory; the real project also has `ktorLibs.server.config.yaml` | Trust the project's own `build.gradle.kts`, and ask IntelliJ's autocomplete for accessor names |

**Principles:** a test that fails after a deliberate behaviour change is a stale test, but always check which of the two it is. Decide each client-error response explicitly and keep 500 for real bugs. Validate once, at the edge.

## Quiz and mistakes
No quiz today. From earlier: 400 = the request itself is malformed, 404 = valid request with nothing behind it.

## English
Words: **latency** (the delay before a response), **throughput** (the work done per second). Grammar: past simple vs present perfect.
✏️ Three sentences about today's topic using both words, one in the past simple and one in the present perfect.

## CTO lens
✏️ What is the business impact of API contracts (cost, hiring, build vs buy, risk)? Invest now or later? Shape: *Decision: ___. Because: ___. We will review it when: ___.*

## In my own words
✏️ Why can't I fix a breaking API change from the server side?

## Next
**AI-03** (16 Oct): backpropagation, building a mini autograd engine. **PRJ-03** (18 Oct): Postgres replaces the in-memory store.